In [1]:
#!pip install mlflow
import pandas as pd
import numpy as np

#NETTOYAGE
import re
import string
import nltk
"""nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')"""
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#FEATURES
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
#BERT
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import mlflow.tensorflow

#TRACKING
import mlflow
import mlflow.sklearn

# Importation du jeu

In [ ]:
columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv("Data/training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)

print(df["tweet"][df["tweet"].isnull()])

print(df.head())
print(df)

# Nettoyage du texte

## Fonction

In [ ]:
#Fonction pour nettoyer le texte
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    
    #Passer en minuscule tout le texte
    texte = texte.lower()

    #Supprimer des éléments frequents dans des tweets, mais jugés ininteressants pour l'analyse
    texte = re.sub(r'<.*?>', '', texte)
    texte = re.sub(r'@\w+', '', texte)
    texte = re.sub(r'\d+', '', texte)
    texte = re.sub(r'https?://\S+|www\.\S+', '', texte)

    #Supprimer les ponctuactions
    texte = texte.translate(str.maketrans("","",string.punctuation))
    
    texte = re.sub(r'\s+', ' ', texte)

    #Initialisation
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    #TOKENISATION
    tokens = texte.split()

    #Parcours des tokens:
    clean_tokens = [
        lemmatizer.lemmatize(token) 
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
            
    #Renvoie des elements à joindre
    return " ".join(clean_tokens)



## Application

In [ ]:
df_trt = df.copy()[['TARGET', 'id', 'date','user', 'tweet']]

df_trt['tweet_net'] = df_trt['tweet'].apply(nettoyer_texte)
print(df_trt['tweet'], df_trt['tweet_net'])

# Featuring